## NB For Testing / Recording Bitboard Implementation

Actual implementation is made in Cython file tablut.pyx

Board representation is 2 quad words in struct 'Bitboard'. First 63 bits (0-62) of Bitboard.low represents the 'bottom 7' ranks, first 18 bits (0-17) of Bitboard.high represents the 'top 2' ranks. I used this representation as it was the approach taken in the most sophisticated 9x9 open source board game implementation I readily found - Shogi program [Apery](https://github.com/HiraokaTakuya/apery_rust/blob/master/src/bitboard.rs). I believe this also affords us the elegant solution of merging the high and low boards with a simple bitwise OR function when creating a key for move generation.

### Checklist
__Groundwork__
- Decide on architecture and define it in a struct
- Write bit manipulation macros that work across both quad words
- Write basic board setting / clearing / printing functions

__Move Generation__
- Calculate moves 'on the fly'
- Pre-calculate blocker masks for all squares
- Generate keys from blocker masked board states
- Use Carry-Rippler trick to generate all possible blocker states for each square
- Use brute force trial and error to find magic numbers
- create a move table for index / move pairs

__Move Making__
- Create multiple Bitboards to track pieces
- Mask occupied Bitboard with move mask for legal moves

__Attacks__
- TBD (mask two over allies, map this to attacked squares, AND this with opponents for attacks?)

In [1]:
import tablut

In [2]:
board = tablut.create_start_position()
tablut.print_board(board)

9 . . . 1 1 1 . . . 
8 . . . . 1 . . . . 
7 . . . . 1 . . . . 
6 1 . . . 1 . . . 1 
5 1 1 1 1 1 1 1 1 1 
4 1 . . . 1 . . . 1 
3 . . . . 1 . . . . 
2 . . . . 1 . . . . 
1 . . . 1 1 1 . . . 
  a b c d e f g h i
Bitboard: 297870862030217272 28688
Bitboard (hex): 0x04223FF888402038 0x0000000000007010



In [3]:
tablut.print_blocker_bits()

14, 13, 13, 13, 13, 13, 13, 13, 14, 
13, 12, 12, 12, 12, 12, 12, 12, 13, 
13, 12, 12, 12, 12, 12, 12, 12, 13, 
13, 12, 12, 12, 12, 12, 12, 12, 13, 
13, 12, 12, 12, 12, 12, 12, 12, 13, 
13, 12, 12, 12, 12, 12, 12, 12, 13, 
13, 12, 12, 12, 12, 12, 12, 12, 13, 
13, 12, 12, 12, 12, 12, 12, 12, 13, 
14, 13, 13, 13, 13, 13, 13, 13, 14, 


In [ ]:
cdef uint64_t Magic[81] = [
0x10400002C200C300,
0x1220000400004040,
0x00800200A4108080,
0x00C0008083000020,
0x00400080A88000B0,
0x6240002040020810,
0x1080010C20080628,
0x02C0022002860204,
0x0040000202020202,
0x3091800204100206,
0x0080400204000081,
0x0100800102448303,
0x0102200100008009,
0x0A03400191080020,
0x1400100040020004,
0x0200100020020002,
0x2000A01000100002,
***Failed to find Magic for square 17***
0x0000000000000000,
0x0008089000020011,
0x1A20084001100419,
0x4240363001001080,
0x0402402000810040,
0x08001018006000D0,
0x00C0405000400008,
0x8080002000102008,
0x0040021000080044,
0x00400050000A0286,
0x1040000810008082,
0x0020004248010005,
0x0020042010004080,
0x0004000088008001,
0x0081050018004001,
0x0080120070001050,
0x0080005810002008,
0x1400400008040004,
0x0140000808000082,
0x080A100002040001,
0x6828000800460080,
0x1002000100010080,
0xD000040100040019,
0x8040210000240020,
0x0102001000940008,
0x42400240000C0008,
0x0940004000040002,
0x0040010AA2040012,
0x8010200800008A00,
0x4050000811000700,
0x9000020006400480,
0x4C00040000800201,
0x1010401004008004,
0x4500008000400201,
0x0100008000400402,
0x4000400020000201,
***Failed to find Magic for square 53***
0x0000000000000000,
0x0410200104008402,
0x1020000400008201,
0x020A022012000102,
0x080800B290800201,
0x2901201080044001,
0x00800200C1400006,
0x4500061C40082004,
0x1800224501082008,
0x0880050420240102,
0x044000080007004F,
0x00E0080015000100,
0x090004240802092C,
0x2640011001000C20,
0x0100020400401044,
0x224000D000200090,
0x014000800090000B,
0x04C004C000180001,
0x9080041060090122,
0x0190200001004100,
0x0246100012000040,
0x1000280000400020,
0x20704001082060C0,
0x0220A00040800010,
0x1880600064200010,
0x1000A00060002C28,
0x0011400040048124,
0x0040200002420052,
]

In [3]:
tablut.test_toggle()

9 . . . 1 1 1 . . . 
8 . . . . 1 . . . . 
7 . . . . 1 . . . . 
6 1 . . . 1 . . . 1 
5 1 1 1 1 1 1 1 1 1 
4 1 . . . 1 . . . 1 
3 . . . . 1 . . . . 
2 . . . . 1 . . . . 
1 . . . 1 1 1 . . . 
  a b c d e f g h i
Bitboard: 297870862030217272 28688
Bitboard (hex): 0x04223FF888402038 0x0000000000007010

9 . . . 1 1 1 . . . 
8 . . . . 1 . . . . 
7 . . . . 1 . . . . 
6 1 . . . 1 . . . 1 
5 1 1 1 1 1 1 1 1 1 
4 1 . . . 1 . . . 1 
3 . . . . 1 . . . . 
2 . . . . 1 . . . . 
1 1 . . 1 1 1 . . . 
  a b c d e f g h i
Bitboard: 297870862030217273 28688
Bitboard (hex): 0x04223FF888402039 0x0000000000007010

9 . . . 1 1 1 . . . 
8 . . . . 1 . . . . 
7 . . . . 1 . . . . 
6 1 . . . 1 . . . 1 
5 1 1 1 1 1 1 1 1 1 
4 1 . . . 1 . . . 1 
3 . . . . 1 . . . . 
2 . . . . 1 . . . . 
1 1 . . 1 . 1 . . . 
  a b c d e f g h i
Bitboard: 297870862030217257 28688
Bitboard (hex): 0x04223FF888402029 0x0000000000007010

9 . . . 1 1 1 . . . 
8 . . . . 1 . . . . 
7 . . . . 1 . . . . 
6 1 . . . 1 . . . 1 
5 1 1 1 1 1 1 1 1 1

apery:
    0x8080_4c00_2020_0191

best yet:
17
0x0060202002000202
28 collisions

0x0140202002040002
40 collisions

0x0100202002040402
40 collisions

0x0100202002040002
40 collisions

0x0000040410286B05
72 collisions

0x044C1000F810AE86
72 collisions

0x004C1000F810AE86
72 collisions

0x004C1000D810B686
72 collisions

0x004C1000C800CB86
72 collisions

0x004C10008800D386
92 collisions

0x000006000429040D
128 collisions

0xF7FFFEFD90010101
171 collisions

0x881000805AF20198
256 collisions

0xEFFFBFFF98015764
256 collisions

0x010010804A57F008
256 collisions

0x08908080080B1008
256 collisions

0x000300085226A630
256 collisions

0x1002800D61485208
256 collisions

0xDFF5EBFFE9536674
256 collisions

0x018080003E14CCB8
256 collisions

0x7F61EFFF88106C00
256 collisions

0x204C820018020400
256 collisions

0x0010140041D60140
260 collisions

0xF9F3FEFF80B400B0
278 collisions

0x10004094125E8A80
340

0x041080205121606C
418

0x108020A052100068
440 collisions

0x0828401008001832
464 collisions

53
0x1000408100F06002

In [ ]:
cdef uint64_t Magic[81] = {
0x0040001141090900,
0x10A0000A00044080,
0x04C004000080A080,
0x00800104102744C0,
0x008000860A122020,
0x4380004500200410,
0x4280010010040908,
0x0040004808080044,
0x804000088800889A,
0x0010200001020001,
0x006110000D000880,
0x9000280100400080,
0x9102200200004001,
0x2800900080020008,
0x0000A00080001001,
0x0200100800100002,
0x1500601000080004,
0x0100440022084012,  # square 17
0x8010145000040900,
0x0430020800121040,
0x4444106001010006,
0x0C04001000440040,
0x14C400200100220A,
0x2682001000200265,
0x00800C2000420808,
0x3440041002080004,
0x41C0001000020082,
0xA010205228000300,
0x0020040024000080,
0x0591042820004801,
0x0004002088008001,
0x021400085000A042,
0x3400004004020008,
0x10800900B0008808,
0x0400400008040004,
0x094000C828000082,
0x121A100002440001,
0x0083100042080041,
0x1000080080010020,
0x82C0020008440040,
0xA002001000240020,
0x0102001000140008,
0x0200100010020004,
0x0040042000340004,
0x0040000504040022,
0x0020001042009200,
0x4020080001004900,
0x1000080100400080,
0x4C00040000800201,
0x2020081010008004,
0x0500008000400201,
0x0100010040000402,
0x2000800010000601,
0x001001010000800A,  # square 53
0x0020007000090045,
0x4000A0B841013402,
0x210830010A001402,
0x0944030420028104,
0x000400100840005B,
0x2002001000200005,
0x00800080004048D2,
0x13220440C8184028,
0x4028998110120204,
0x034000840C000C69,
0x00A0040004000080,
0x0080028801081080,
0x4040010100040040,
0x0240010000240020,
0x4140010000901001,
0x30800046C0002102,
0x00C0004000180009,
0x204000084244000A,
0x0004200001252900,
0x0504900002000840,
0x000A200101000280,
0x0009200085000060,
0x81012000A44000A0,
0x840A400100101110,
0x0681200210200024,
0x8006200210000904,
0xA000200001224282,
}

In [ ]:
cdef uint64_t Magic[81] = {
0x804000C400808100,
0x8220002101000480,
0x4180011A00820080,
0x0140008091000120,
0x00C0020000903010,
0x8040030002105408,
0x0080030608081808,
0x0040004808080044,
0x04C0000408108102,
0x1010200001244100,
0x2000100004112040,
0x2008400400004102,
0x5004400082008002,
0x0440200040400020,
0x00C8A000C0040010,
0x8040200410100008,
0x0052201000100002,
0x0A1002000C010841,
0x21C02C9000212100,
0x0108008800320080,
0x8682022001020080,
0x0018002000882082,
0x0024002000222042,
0x0185001002106001,
0x0480002001400888,
0x014000100288008C,
0x0040007000024082,
0x0840060418000100,
0x0221040084000880,
0x4208004108010009,
0x0808002090004182,
0x0084005008001090,
0x4400044488001001,
0x094010C008000498,
0x8180001810009014,
0x02400008A8000082,
0x0812064800040001,
0x4008000400020100,
0x10002002038C0080,
0x9244000200040011,
0x01020E000108000A,
0x0000850008040048,
0x04400040000C0008,
0x0040004004240022,
0x14400048020C0032,
0x0308081104001200,
0x0050011200120100,
0x3002000400800080,
0x0C00020001000201,
0x0200020000400201,
0x1500010000200201,
0x2300008000400402,
0x0000800010000201,
0x03F21010A9040012,
0x80A04082204C0084,
0x4002410808010852,
0x0020080040808406,
0x8408000100480286,
0x0080109080800042,
0x0100040888204004,
0x0480088001440902,
0x440042000B106008,
0x1880120410A00204,
0x0040008800810041,
0x0420003200020080,
0x010002088A008801,
0x02400510008000C0,
0x4040049010800130,
0x1100010243034404,
0x1080098000408802,
0x8A40009000100002,
0x5C80020420042102,
0x0285200021010900,
0xC00018009D000120,
0x1000880000400020,
0x0102A00040800040,
0x6800200020204410,
0x0000400040848830,
0x0600200140002404,
0x840E4000400421C4,
0x4100200004008292,
}

In [2]:
magics = tablut.find_magics()

****FINDING MAGICS****



Square 80: Best 13930/16384:   0%|          | 139932/100000000 [00:01<13:40, 121635.05magic/s]

cdef uint64_t Magic[81] = {
0x804000C400808100,
0x8220002101000480,
0x4180011A00820080,
0x0140008091000120,
0x00C0020000903010,
0x8040030002105408,
0x0080030608081808,
0x0040004808080044,
0x04C0000408108102,
0x1010200001244100,
0x2000100004112040,
0x2008400400004102,
0x5004400082008002,
0x0440200040400020,
0x00C8A000C0040010,
0x8040200410100008,
0x0052201000100002,
0x0A1002000C010841,
0x21C02C9000212100,
0x0108008800320080,
0x8682022001020080,
0x0018002000882082,
0x0024002000222042,
0x0185001002106001,
0x0480002001400888,
0x014000100288008C,
0x0040007000024082,
0x0840060418000100,
0x0221040084000880,
0x4208004108010009,
0x0808002090004182,
0x0084005008001090,
0x4400044488001001,
0x094010C008000498,
0x8180001810009014,
0x02400008A8000082,
0x0812064800040001,
0x4008000400020100,
0x10002002038C0080,
0x9244000200040011,
0x01020E000108000A,
0x0000850008040048,
0x04400040000C0008,
0x0040004004240022,
0x14400048020C0032,
0x0308081104001200,
0x0050011200120100,
0x3002000400800080,
0x0C00020001

In [3]:
blockers, moves = tablut.test_magic(7, 0x0040004808080044)

move 0 successfully hashed to index 0
move 1 successfully hashed to index 16
move 2 successfully hashed to index 32
move 3 successfully hashed to index 48
move 4 successfully hashed to index 64
move 5 successfully hashed to index 80
move 6 successfully hashed to index 96
move 7 successfully hashed to index 112
move 8 successfully hashed to index 128
move 9 successfully hashed to index 144
move 10 successfully hashed to index 160
move 11 successfully hashed to index 176
move 12 successfully hashed to index 192
move 13 successfully hashed to index 208
move 14 successfully hashed to index 224
move 15 successfully hashed to index 240
move 16 successfully hashed to index 256
move 17 successfully hashed to index 272
move 18 successfully hashed to index 288
move 19 successfully hashed to index 304
move 20 successfully hashed to index 320
move 21 successfully hashed to index 336
move 22 successfully hashed to index 352
move 23 successfully hashed to index 368
move 24 successfully hashed to ind

In [53]:
blockers[4130]

35072

In [54]:
moves[4130]

{'low': 4620710844295086336, 'high': 0}

In [55]:
tablut.py_build_board(35072, 0)

9 . . . . . . . . . 
8 . . . . . . . . . 
7 . . . . . . . . . 
6 . . . . . . . . . 
5 . . . . . . . . . 
4 . . . . . . . . . 
3 . . . . . . . . . 
2 . . 1 . . . 1 . . 
1 . . . . . . . . 1 
  a b c d e f g h i
Bitboard: 35072 0
Bitboard (hex): 0x0000000000008900 0x0000000000000000



In [57]:
tablut.py_build_board(4620710844295086336, 0)

9 . . . . . . . . . 
8 . . . . . . . . . 
7 . . . . . . . . 1 
6 . . . . . . . . 1 
5 . . . . . . . . 1 
4 . . . . . . . . 1 
3 . . . . . . . . 1 
2 . . . . . . . 1 . 
1 . . . . . . . . 1 
  a b c d e f g h i
Bitboard: 4620710844295086336 0
Bitboard (hex): 0x4020100804010100 0x0000000000000000



In [85]:
tablut.py_build_board(17626545783040, 0)

9 . . . . . . . . . 
8 . . . . . . . . . 
7 . . . . . . . . . 
6 . . . . . . . . . 
5 . . . . . . . . 1 
4 . . . . . . . . 1 
3 . . . . . . . . . 
2 . . . . . . . . . 
1 . . . . . . . . 1 
  a b c d e f g h i
Bitboard: 17626545783040 0
Bitboard (hex): 0x0000100800000100 0x0000000000000000



In [86]:
tablut.py_build_board(67239680, 0)

9 . . . . . . . . . 
8 . . . . . . . . . 
7 . . . . . . . . . 
6 . . . . . . . . . 
5 . . . . . . . . . 
4 . . . . . . . . . 
3 . . . . . . . . 1 
2 1 1 1 1 1 1 1 1 . 
1 . . . . . . . . 1 
  a b c d e f g h i
Bitboard: 67239680 0
Bitboard (hex): 0x000000000401FF00 0x0000000000000000



In [5]:
2**14

16384

In [21]:
blockers, moves = tablut.py_build_bm_tables(17)

unique_moves = []
for move in moves:
    if move not in unique_moves:
        unique_moves.append(move)
print(f"out of {len(moves)} total moves, {len(unique_moves)} unique moves")

out of 8192 total moves, 56 unique moves


In [5]:
2**14

16384

In [2]:
tablut.check_subsets()

9 . . . . . . . . . 
8 . . . . . . . . . 
7 . . . . . . . . . 
6 . . . . . . . . . 
5 . . . . . . . . . 
4 . . . . . . . . . 
3 . . . . . . . . . 
2 . . . . . . . . . 
1 . . . . . . . . . 
  a b c d e f g h i
Bitboard: 0 0
Bitboard (hex): 0x0000000000000000 0x0000000000000000

9 . . . . . . . . . 
8 . . . . . . . . . 
7 . . . . . . . . . 
6 . . . . . . . . . 
5 . . . . . . . . . 
4 . . . . . . . . . 
3 . . . . . . . . . 
2 . . . . . 1 . . . 
1 . . . . . . . . . 
  a b c d e f g h i
Bitboard: 16384 0
Bitboard (hex): 0x0000000000004000 0x0000000000000000

9 . . . . . . . . . 
8 . . . . . . . . . 
7 . . . . . . . . . 
6 . . . . . . . . . 
5 . . . . . . . . . 
4 . . . . . . . . . 
3 . . . . . 1 . . . 
2 . . . . . . . . . 
1 . . . . . . . . . 
  a b c d e f g h i
Bitboard: 8388608 0
Bitboard (hex): 0x0000000000800000 0x0000000000000000

9 . . . . . . . . . 
8 . . . . . . . . . 
7 . . . . . . . . . 
6 . . . . . . . . . 
5 . . . . . . . . . 
4 . . . . . . . . . 
3 . . . . . 1 . . . 
2 . . . .

In [13]:
2 ** 12

4096

In [4]:
moves = tablut.find_moves_otf(start_pos, 77)
moves.print_board()

9 . . . . . . 1 1 1 
8 . . . . . 1 . . . 
7 . . . . . 1 . . . 
6 . . . . . 1 . . . 
5 . . . . . . . . . 
4 . . . . . . . . . 
3 . . . . . . . . . 
2 . . . . . . . . . 
1 . . . . . . . . . 
  a b c d e f g h i
Bitboard: 577586652210266112 229408
Bitboard (hex): 0x0804000000000000 0x0000000000038020



In [2]:
tablut.generate_attack_masks()

BitboardStruct(low = 0x00402010080403FE, high = 0x0000000000000201),
BitboardStruct(low = 0x00804020100805FD, high = 0x0000000000000402),
BitboardStruct(low = 0x01008040201009FB, high = 0x0000000000000804),
BitboardStruct(low = 0x02010080402011F7, high = 0x0000000000001008),
BitboardStruct(low = 0x04020100804021EF, high = 0x0000000000002010),
BitboardStruct(low = 0x08040201008041DF, high = 0x0000000000004020),
BitboardStruct(low = 0x10080402010081BF, high = 0x0000000000008040),
BitboardStruct(low = 0x201008040201017F, high = 0x0000000000010080),
BitboardStruct(low = 0x40201008040200FF, high = 0x0000000000020100),
BitboardStruct(low = 0x004020100807FC01, high = 0x0000000000000201),
BitboardStruct(low = 0x00804020100BFA02, high = 0x0000000000000402),
BitboardStruct(low = 0x010080402013F604, high = 0x0000000000000804),
BitboardStruct(low = 0x020100804023EE08, high = 0x0000000000001008),
BitboardStruct(low = 0x040201008043DE10, high = 0x0000000000002010),
BitboardStruct(low = 0x08040201008

In [9]:
rank = 3
pos = 0

masked_rank = start_pos.get_rank_key(rank)
_tmp = tablut.Bitboard(masked_rank)
_tmp.print_board()
moves = tablut.get_moves(masked_rank, pos)
moves.print_board()

9 . . . . . . . . . 
8 . . . . . . . . . 
7 . . . . . . . . . 
6 . . . . . . . . . 
5 . . . . . . . . . 
4 . . . . . . . . . 
3 . . . . . . . . . 
2 . . . . . . . . . 
1 1 . . . 1 . . . 1 
  a b c d e f g h i
Bitboard: 273 0
Bitboard (hex): 0x0000000000000111 0x0000000000000000

9 . . . . . . . . . 
8 . . . . . . . . . 
7 . . . . . . . . . 
6 . . . . . . . . . 
5 . . . . . . . . . 
4 . . . . . . . . . 
3 . . . . . . . . . 
2 . . . . . . . . . 
1 . 1 1 1 . . . . . 
  a b c d e f g h i
Bitboard: 14 0
Bitboard (hex): 0x000000000000000E 0x0000000000000000



In [18]:
target_rank = 3
start_pos_move_key = start_pos.shift_rank_to_bottom(target_rank)
start_pos_move_key.print_board()

9 . . . . . . . . . 
8 . . . . . . . . . 
7 . . . . . . . . . 
6 . . . . . . . . . 
5 . . . . . . . . . 
4 . . . . . . . . . 
3 . . . . . . . . . 
2 . . . . . . . . . 
1 1 . . . 1 . . . 1 
  a b c d e f g h i
Bitboard: 273 0
Bitboard (hex): 0x0000000000000111 0x0000000000000000



In [9]:
tmp = tablut.Bitboard()

for i in range(36, 45):
    tmp.set_bit(i)

tmp.print_board()

9 . . . . . . . . . 
8 . . . . . . . . . 
7 . . . . . . . . . 
6 . . . . . . . . . 
5 1 1 1 1 1 1 1 1 1 
4 . . . . . . . . . 
3 . . . . . . . . . 
2 . . . . . . . . . 
1 . . . . . . . . . 
  a b c d e f g h i
Bitboard: 35115652612096 0
Bitboard (hex): 0x00001FF000000000 0x0000000000000000



In [10]:
_tmp = tmp.shift_rank_to_bottom(4)
_tmp.print_board()

9 . . . . . . . . . 
8 . . . . . . . . . 
7 . . . . . . . . . 
6 . . . . . . . . . 
5 . . . . . . . . . 
4 . . . . . . . . . 
3 . . . . . . . . . 
2 . . . . . . . . . 
1 1 1 1 1 1 1 1 1 1 
  a b c d e f g h i
Bitboard: 511 0
Bitboard (hex): 0x00000000000001FF 0x0000000000000000



In [25]:
piece_pos = 4
moves = tablut.get_moves(start_pos_move_key, piece_pos)
moves.print_board()

9 . . . . . . . . . 
8 . . . . . . . . . 
7 . . . . . . . . . 
6 . . . . . . . . . 
5 . . . . . . . . . 
4 . . . . . . . . . 
3 . . . . . . . . . 
2 . . . . . . . . . 
1 . 1 1 1 . 1 1 1 . 
  a b c d e f g h i
Bitboard: 238 0
Bitboard (hex): 0x00000000000000EE 0x0000000000000000



In [3]:
board.print_board()

9 . . . 1 1 1 . . . 
8 . . . . 1 . . . . 
7 . . . . 1 . . . . 
6 1 . . . 1 . . . 1 
5 1 1 1 1 1 1 1 1 1 
4 1 . . . 1 . . . 1 
3 . . . . 1 . . . . 
2 . . . . 1 . . . . 
1 . . . 1 1 1 . . . 
  a b c d e f g h i
Bitboard: 297870862030217272 57376
Bitboard (hex): 0x04223FF888402038 0x000000000000E020



In [4]:
class ULL64:
    def __init__(self, value=0):
        self.value = value & ((1 << 64) - 1)  # Ensure 64-bit

    def __and__(self, other):
        return ULL64(self.value & other.value)

    def __or__(self, other):
        return ULL64(self.value | other.value)

    def __xor__(self, other):
        return ULL64(self.value ^ other.value)

    def __lshift__(self, shift):
        return ULL64(self.value << shift)

    def __rshift__(self, shift):
        return ULL64(self.value >> shift)

    def __invert__(self):
        return ULL64(~self.value & ((1 << 64) - 1))

    def __str__(self):
        return f"{self.value:064b}"

    def __repr__(self):
        return f"ULL64({self.value})"

    def __sub__(self, other):
        return ULL64(self.value - other.value)

    def bit_length(self):
        return self.value.bit_length()

    def set_bit(self, index):
        if 0 <= index < 64:
            self.value |= (1 << index)
        else:
            raise ValueError("Bit index must be between 0 and 63")

    def clear_bit(self, index):
        if 0 <= index < 64:
            self.value &= ~(1 << index)
        else:
            raise ValueError("Bit index must be between 0 and 63")

    def is_bit_set(self, index):
        if 0 <= index < 64:
            return bool(self.value & (1 << index))
        else:
            raise ValueError("Bit index must be between 0 and 63")

class Bitboard:
    def __init__(self, low=0, high=0):
        self.low = ULL64(low)
        self.high = ULL64(high)

    def set_bit(self, index):
        if 0 <= index < 62:
            self.low.set_bit(index)
        elif 62 <= index < 81:
            self.high.set_bit(index - 62)
        else:
            raise ValueError("Bit index must be between 0 and 80")

    def clear_bit(self, index):
        if 0 <= index < 62:
            self.low.clear_bit(index)
        elif 62 <= index < 81:
            self.high.clear_bit(index - 62)
        else:
            raise ValueError("Bit index must be between 0 and 80")

    def is_bit_set(self, index):
        if 0 <= index < 62:
            return self.low.is_bit_set(index)
        elif 62 <= index < 81:
            return self.high.is_bit_set(index - 62)
        else:
            raise ValueError("Bit index must be between 0 and 80")


    def clear_all(self):
        self.low.value = 0
        self.high.value = 0

    def print_board(self):
        for rank in range(8, -1, -1):
            print(f"{rank+1} ", end="")
            for file in range(9):
                sq = rank * 9 + file
                if sq < 62:
                    print("1 " if self.low.is_bit_set(sq) else ". ", end="")
                else:
                    print("1 " if self.high.is_bit_set(sq - 62) else ". ", end="")
            print()
        print("  a b c d e f g h i")
        print(f"Bitboard: {self.low.value} {self.high.value}")
        print(f"Bitboard (hex): 0x{self.low.value:016X} 0x{self.high.value:016X}\n")


In [10]:
import time

In [11]:
def generate_moves(board):
    key_list = []
    value_list = []
    for i in range(9):
        moves = ULL64()
        if board.is_bit_set(i):
            for j in range(1, 8):
                if i + j < 9:
                    if not board.is_bit_set(i + j):
                        moves.set_bit(i + j)
                    else:
                        break
                else:
                    break
            for j in range(1, 8):
                if i - j >= 0:
                    if not board.is_bit_set(i - j):
                            moves.set_bit(i - j)
                    else:
                        break
                else:
                    break
            key_list.append((board.low.value, i))
            value_list.append(moves)
    return key_list, value_list

def generate_combinations():
    key_list = []
    value_list = []
    board = Bitboard()
    for i in range(512):
        print(f"{i+1}/512 Positions Added", end='\r')
        board.clear_all()  # Clear the board before setting new bits
        for j in range(9):
            if i & (1 << j):
                board.set_bit(j)
        keys, values = generate_moves(board)
        key_list += keys
        value_list += values
    return key_list, value_list

keys, values = generate_combinations()
print('\njoining keys and values...')
move_table = dict(zip(keys, values))
print('Move table created')


KeyboardInterrupt: 

In [183]:
len(move_table)

2304

In [ ]:
def generate_attack_table():


In [8]:
2^14 * 4 + 2**13 * 28 + 2**12 *49

430138